In [ ]:
%%bash
# FOLIO_CELL_SETUP: clone the speed-tier branch and build the Studio runtime once per VM
cd /content
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo NO_GPU
[ -d folio ] || git clone -q -b perf/long-takes-speed-tiers https://github.com/adamholter/folio folio
(cd folio && git log --oneline -1)
if [ ! -x ~/.local/share/folio/f-venv/bin/python ] || [ ! -f /content/.folio_setup_ok ]; then
  python3 folio/tools/setup_studio_runtime.py > setup.log 2>&1 && touch /content/.folio_setup_ok
  tail -3 setup.log
fi
which ffmpeg >/dev/null || (apt-get -qq update && apt-get -qq install -y ffmpeg) >/dev/null 2>&1
[ -f /content/.folio_setup_ok ] && echo FOLIO_SETUP_OK || echo FOLIO_SETUP_FAILED


In [ ]:
# FOLIO_CELL_UPLOAD: pick the FLAC inputs (driven by colab_run.sh)
import os, shutil
from google.colab import files
shutil.rmtree('/content/in', ignore_errors=True); os.makedirs('/content/in')
os.chdir('/content/in')
up = files.upload()
os.chdir('/content')
print('FOLIO_UPLOAD_OK', len(up), sorted(up))


In [ ]:
%%bash
# FOLIO_CELL_PROCESS: Folio Studio --device cuda --quality high on every uploaded FLAC
cd /content; rm -rf out out.zip; mkdir -p out
t0=$(date +%s.%N)
for f in in/*.flac; do
  n=$(basename "$f" .flac)
  ffmpeg -nostdin -v error -y -i "$f" -c:a pcm_s24le "in/$n.wav"
  a=$(ffprobe -v error -show_entries format=duration -of csv=p=0 "in/$n.wav")
  s=$(date +%s.%N)
  folio/bin/folio-studio --input "in/$n.wav" --output "out/$n.wav" --device cuda --quality high > "out/$n.log" 2>&1; rc=$?
  w=$(echo "$(date +%s.%N) - $s" | bc)
  if [ $rc -eq 0 ] && [ -f "out/$n.wav" ]; then ffmpeg -nostdin -v error -y -i "out/$n.wav" -c:a flac "out/$n.flac" && rm "out/$n.wav"; fi
  echo "FOLIO_RESULT $n rc=$rc audio=$a wall=$w"
  echo "$n rc=$rc audio=$a wall=$w" >> out/results.txt
done
echo "total_wall=$(echo "$(date +%s.%N) - $t0" | bc)" >> out/results.txt
(cd out && zip -q -0 ../out.zip *)
echo FOLIO_PROCESS_DONE


In [ ]:
# FOLIO_CELL_DOWNLOAD: send results back to the driving browser
from google.colab import files
files.download('/content/out.zip')
print('FOLIO_DOWNLOAD_SENT')
